# Gaussian Copula

First of the four generation notebooks. Each follows the same shape: load the shared
training data, fit the generator, produce a synthetic dataset of the same size, save it,
and record how long training and generation took. One method per notebook means each gets
identical treatment, and a problem with one cannot interrupt the others.

Gaussian Copula is the statistical baseline. It learns the shape of each column and the
correlations between them, then samples from those shapes in a way that respects the
correlations. No neural networks are involved, which makes it much the fastest of the four
and a useful reference point: if the more complex methods cannot beat it, their extra cost
is hard to justify.

Its known limitation, covered in the literature review, is that correlation only captures
straight-line relationships. More complicated interactions can be missed, and the
evaluation will show whether that matters for this cohort.

## Setup

Gaussian Copula comes from sdv, which also provides CTGAN and TVAE. No GPU is needed;
this method fits in seconds.

In [1]:
%pip install -q pandas pyarrow sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.9/209.9 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 143.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.0/207.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.6 MB/s eta 0:00:00


## Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and
delete the data once the work is finished.

In [2]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

Mounted at /content/drive
Working folder set to Google Drive: /content/drive/MyDrive/mimic-synthetic-pipeline


## Training data

Only the training set is loaded. The generator must never see the test set, which is what
the evaluation later uses to judge whether synthetic-trained models generalise to unseen
patients.

In [3]:
from pathlib import Path

import pandas as pd

if not Path("data/train.parquet").exists():
    raise FileNotFoundError(
        "data/train.parquet not found. Run the extraction and preparation steps first."
    )

TARGET = "readmitted_30d"
train_df = pd.read_parquet("data/train.parquet")
print(f"Training data: {len(train_df):,} admissions, readmission rate {train_df[TARGET].mean():.4f}")

Training data: 427,408 admissions, readmission rate 0.2067


## Fitting and generating

The model is fitted on the training data, then asked for a synthetic dataset with the
same number of rows. Matching the size keeps the comparison simple: every method produces
one synthetic training set of identical size.

Training and generation are timed and appended to a shared log. The four methods differ
sharply in computational cost, and the timings turn that into a measured result rather
than an assertion.

In [4]:
import time

from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer

metadata = Metadata.detect_from_dataframe(train_df, table_name="cohort")
model = GaussianCopulaSynthesizer(metadata)

t0 = time.time()
model.fit(train_df)
train_seconds = time.time() - t0

t0 = time.time()
synthetic_df = model.sample(num_rows=len(train_df))
generate_seconds = time.time() - t0

synthetic_df.to_parquet("data/synthetic_gaussian_copula.parquet", index=False)
print(f"Training took {train_seconds:.1f}s, generation took {generate_seconds:.1f}s")
print(f"Saved {len(synthetic_df):,} synthetic admissions to data/synthetic_gaussian_copula.parquet")

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Training took 105.5s, generation took 7.3s
Saved 427,408 synthetic admissions to data/synthetic_gaussian_copula.parquet


## Sanity checks

Not the full evaluation, just quick aggregate checks that the generator produced
something sensible. The readmission rate should sit near the real training rate of about
0.21, and the age and length-of-stay averages should be plausible. Catching an obviously
broken run here saves a wasted evaluation later.

In [5]:
checks = pd.DataFrame({
    "statistic": ["Readmission rate", "Mean age", "Mean length of stay (days)"],
    "real_training_data": [
        round(train_df[TARGET].mean(), 4),
        round(train_df["age_at_admission"].astype(float).mean(), 1),
        round(train_df["length_of_stay_days"].mean(), 2),
    ],
    "synthetic_data": [
        round(synthetic_df[TARGET].mean(), 4),
        round(synthetic_df["age_at_admission"].astype(float).mean(), 1),
        round(synthetic_df["length_of_stay_days"].mean(), 2),
    ],
})
checks

,statistic,real_training_data,synthetic_data
0,Readmission rate,0.2067,0.2088
1,Mean age,58.8000,59.6000
2,Mean length of stay (days),4.6400,4.2800


In [6]:
# Append this run's timings to the shared generation log used by all four methods.
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
log_path = out_dir / "generation_log.csv"

entry = pd.DataFrame([{
    "method": "Gaussian Copula",
    "rows_generated": len(synthetic_df),
    "train_seconds": round(train_seconds, 1),
    "generate_seconds": round(generate_seconds, 1),
}])
if log_path.exists():
    log = pd.read_csv(log_path)
    log = log[log["method"] != "Gaussian Copula"]
    log = pd.concat([log, entry], ignore_index=True)
else:
    log = entry
log.to_csv(log_path, index=False)
log

,method,rows_generated,train_seconds,generate_seconds
0,TVAE,427408,453.1,3.1
1,TabDDPM,427408,6153.8,83.4
2,CTGAN,427408,6724.9,5.5
3,Gaussian Copula,427408,105.5,7.3
